# DataJam5 Bogotá 2026: desabastecimiento de medicamentos y mortalidad prematura por enfermedades crónicas
**Equipo:** TINTOS · **Entrega:** 16 octubre 2026

**Pregunta:** ¿los grupos de enfermedad crónica con mayor mortalidad prematura en Bogotá son también los que enfrentan más desabastecimiento de medicamentos prioritarios?

**Hipótesis**
- H1: más mortalidad prematura → más medicamentos prioritarios desabastecidos.
- H2: los desabastecimientos prioritarios duran más que los no prioritarios (requiere fecha de cierre).
- H3: las localidades con más mortalidad prematura son prioritarias para el monitoreo.

**Alcance:** Bogotá, adultos de 60 a 69 años. Lectura distrital, por localidad, por sexo y en el tiempo.

**Fuentes (Portal de Datos Abiertos Bogotá):** desabastecimiento de medicamentos y mortalidad prematura por cardiocerebrovascular, neoplasias, respiratorias crónicas y diabetes.

**Ejecutar:** *Entorno de ejecución → Ejecutar todo*.

## 1. Configuración y carga

In [1]:
import re, unicodedata, os, warnings, io, time
import requests
import numpy as np
import pandas as pd
import plotly.express as px
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)   # sin recortar textos con "..."

# Nombres técnicos -> nombres legibles (tablas, Excel y CSV de salida)
BONITO = {"anio": "año", "anio_ref": "año_ref", "nombre_del_farmaco": "fármaco",
          "indicaciony_o_tratamiento": "indicación o tratamiento", "prioridad_en_salud": "prioridad en salud",
          "muertes_60_69": "muertes (60-69 años)"}

def tabla(df, ancho=380):
    # Tabla ordenada: texto a la izquierda, números a la derecha, celdas amplias, texto largo en varias líneas
    try:
        d = (df.to_frame() if isinstance(df, pd.Series) else df).rename(columns=BONITO)
        est = d.style.hide(axis="index")
        est = est.set_properties(**{"text-align": "left", "white-space": "normal", "vertical-align": "top",
                                    "padding": "8px 16px", "max-width": f"{ancho}px", "font-size": "14px"})
        est = est.set_table_styles([
            {"selector": "", "props": [("border-collapse", "collapse")]},
            {"selector": "th", "props": [("text-align", "left"), ("padding", "10px 16px"), ("background-color", "#f1f3f5"),
                                         ("color", "#212529"), ("border-bottom", "2px solid #ced4da")]},
            {"selector": "td", "props": [("border-bottom", "1px solid #dee2e6")]},
        ])
        num = list(d.select_dtypes("number").columns)
        flt = list(d.select_dtypes("float").columns)
        if num:
            est = est.set_properties(subset=num, **{"text-align": "right"})
        if flt:
            est = est.format(precision=1, subset=flt)
        display(est)
    except Exception:
        display(df)   # respaldo si Styler no está disponible

# Carpeta de trabajo: Drive en Colab, carpeta actual fuera de Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = "/content/drive/MyDrive/DataJam"
except Exception:
    CARPETA = ".." if os.path.isdir("../data") else "."   # repo: notebook dentro de /notebooks
os.makedirs(CARPETA, exist_ok=True)
os.chdir(CARPETA)
for carpeta in ["data/raw", "data/processed", "outputs"]:
    os.makedirs(carpeta, exist_ok=True)
print("Carpeta:", os.getcwd(), "| CSV en data/raw:", os.listdir("data/raw"))

# Archivos esperados en data/raw
FUENTES = {
    "medicamentos": "data/raw/medicamentos.csv",
    "cardio": "data/raw/cardio.csv",
    "neoplasias": "data/raw/neoplasias.csv",
    "respiratorias": "data/raw/respiratorias.csv",
    "diabetes": "data/raw/diabetes.csv",
}

def leer_csv(origen, nombre):
    # Usa data/raw si existe; si no, descarga (si es URL) y guarda copia
    ruta = f"data/raw/{nombre}.csv"
    if os.path.exists(ruta):
        print(f"   {nombre}: usando data/raw")
    elif str(origen).startswith("http"):
        r = requests.get(origen, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        with open(ruta, "wb") as f:
            f.write(r.content)
        print(f"   descargado {nombre}")
    else:
        ruta = origen
    with open(ruta, "rb") as f:
        raw = f.read()
    for enc in ["utf-8-sig", "latin1"]:
        try:
            texto = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    primera = texto.split("\n", 1)[0]
    sep = max([",", ";", "\t", "|"], key=primera.count)
    return pd.read_csv(io.StringIO(texto), sep=sep, low_memory=False)

datos = {}
for nombre, origen in FUENTES.items():
    try:
        datos[nombre] = leer_csv(origen, nombre)
        print(f"OK     {nombre}: {datos[nombre].shape}")
    except Exception as e:
        print(f"FALLÓ  {nombre}: {str(e)[:100]} -> suba el CSV a data/raw/{nombre}.csv")

Mounted at /content/drive
Carpeta: /content/drive/MyDrive/DataJam | CSV en data/raw: ['medicamentos.csv', 'cardio.csv', 'neoplasias.csv', 'respiratorias.csv', 'diabetes.csv']
   medicamentos: usando data/raw
OK     medicamentos: (237, 8)
   cardio: usando data/raw
OK     cardio: (720, 8)
   neoplasias: usando data/raw
OK     neoplasias: (9894, 8)
   respiratorias: usando data/raw
OK     respiratorias: (1301, 8)
   diabetes: usando data/raw
OK     diabetes: (2371, 8)


### 1.1 Inspección inicial

In [2]:
for n, d in datos.items():
    print("=" * 80)
    print(n, d.shape)
    print(list(d.columns))
    tabla(d.head(3))

medicamentos (237, 8)
['_id', 'mes', 'Nombre del farmaco', 'Estado', 'Causa', 'Fecha de cierre', 'Indicaciony/o tratamiento', 'Prioridad en Salud']


_id,mes,Nombre del farmaco,Estado,Causa,Fecha de cierre,Indicaciony/o tratamiento,Prioridad en Salud
1,Agosto,Abacavir solución oral,Desabastecido***: incluido en el LMVND,Alerta sanitaria / Descontinuación del producto,nan,AGENTES DE ACCION DIRECTA CONTRA EL VIRUS (SISTEM),ENFERMEDADES INFECCIOSAS
2,Agosto,Abacavir tabletas,En monitorización,Baja rentabilidad,nan,ANTIVIRALES DE USO SISTEMICO,ENFERMEDADES INFECCIOSAS
3,Agosto,Acetato de ciproterona tableta,En Riesgo De Desabastecido,Aumento de la demanda,nan,ANTIANDROGENOS,CÁNCER


cardio (720, 8)
['AÑO', 'SEXO', 'NACIONALIDAD', 'EDAD_QUINQUENAL', 'CIE10_BASICA', 'LOCALIDAD', 'CLASIFICACION_CRONICAS', 'CASOS']


AÑO,SEXO,NACIONALIDAD,EDAD_QUINQUENAL,CIE10_BASICA,LOCALIDAD,CLASIFICACION_CRONICAS,CASOS
2026,Femenino,Colombiano,De 30 a 34 años,I219,08 - Kennedy,Enfermedades Cardiocerebrovasculares,1
2026,Femenino,Colombiano,De 30 a 34 años,I219,09 - Fontibón,Enfermedades Cardiocerebrovasculares,1
2026,Femenino,Colombiano,De 30 a 34 años,I219,11 - Suba,Enfermedades Cardiocerebrovasculares,1


neoplasias (9894, 8)
['AÑO', 'SEXO', 'NACIONALIDAD', 'EDAD_QUINQUENAL', 'CIE10_BASICA', 'LOCALIDAD', 'CLASIFICACION_CRONICAS', 'CASOS']


AÑO,SEXO,NACIONALIDAD,EDAD_QUINQUENAL,CIE10_BASICA,LOCALIDAD,CLASIFICACION_CRONICAS,CASOS
2019,Femenino,Colombiano,De 30 a 34 años,C169,04 - San Cristóbal,Neoplasias,2
2019,Femenino,Colombiano,De 30 a 34 años,C169,08 - Kennedy,Neoplasias,1
2019,Femenino,Colombiano,De 30 a 34 años,C169,11 - Suba,Neoplasias,1


respiratorias (1301, 8)
['AÑO', 'SEXO', 'NACIONALIDAD', 'EDAD_QUINQUENAL', 'CIE10_BASICA', 'LOCALIDAD', 'CLASIFICACION_CRONICAS', 'CASOS']


AÑO,SEXO,NACIONALIDAD,EDAD_QUINQUENAL,CIE10_BASICA,LOCALIDAD,CLASIFICACION_CRONICAS,CASOS
2019,Femenino,Colombiano,De 35 a 39 años,J440,10 - Engativá,Enfermedades Cronicas De Las Vias Respiratorias Inferiores,1
2019,Femenino,Colombiano,De 35 a 39 años,J449,18 - Rafael Uribe Uribe,Enfermedades Cronicas De Las Vias Respiratorias Inferiores,1
2019,Femenino,Colombiano,De 35 a 39 años,J46X,07 - Bosa,Enfermedades Cronicas De Las Vias Respiratorias Inferiores,1


diabetes (2371, 8)
['AÑO', 'SEXO', 'NACIONALIDAD', 'EDAD_QUINQUENAL', 'CIE10_BASICA', 'LOCALIDAD', 'CLASIFICACION_CRONICAS', 'CASOS']


AÑO,SEXO,NACIONALIDAD,EDAD_QUINQUENAL,CIE10_BASICA,LOCALIDAD,CLASIFICACION_CRONICAS,CASOS
2019,Femenino,Colombiano,De 30 a 34 años,E119,16 - Puente Aranda,Diabetes,1
2019,Femenino,Colombiano,De 30 a 34 años,E141,08 - Kennedy,Diabetes,1
2019,Femenino,Colombiano,De 30 a 34 años,E146,01 - Usaquén,Diabetes,1


## 2. Limpieza y transformación

In [3]:
def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def norm_col(c):
    c = sin_tildes(c).lower().strip()
    return re.sub(r"[^a-z0-9]+", "_", c).strip("_")

def norm_txt(x):
    return sin_tildes(x).lower().strip() if pd.notna(x) else x

def limpiar(df):
    # Columnas normalizadas, sin duplicados, textos limpios
    df = df.copy()
    df.columns = [norm_col(c) for c in df.columns]
    n0 = len(df)
    df = df.drop_duplicates()
    print(f"   duplicados eliminados: {n0 - len(df)}")
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan, "": np.nan, "None": np.nan})
    return df

def buscar_col(df, *claves):
    for c in df.columns:
        if all(k in c for k in claves):
            return c
    return None

### 2.1 Medicamentos: limpieza y estado
- `Estado` mezcla desabastecido, en riesgo, en monitorización y no desabastecido. Solo el primero es desabastecimiento efectivo.
- `Prioridad en Salud` es una categoría temática, no un sí/no.

In [4]:
print("Medicamentos:")
med = limpiar(datos["medicamentos"])

C = {
    "mes": buscar_col(med, "mes"),
    "farmaco": buscar_col(med, "farmaco") or buscar_col(med, "nombre"),
    "estado": buscar_col(med, "estado"),
    "causa": buscar_col(med, "causa"),
    "cierre": buscar_col(med, "cierre"),
    "indicacion": buscar_col(med, "indicacion"),
    "prioridad": buscar_col(med, "prioridad"),
}
print("Columnas:", C)
faltan = [k for k, v in C.items() if v is None]
if faltan:
    raise ValueError(f"Falta columna para {faltan}")

# Mes a número y fecha de cierre a fecha
MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
         "julio": 7, "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}

def mes_a_num(x):
    if pd.isna(x):
        return np.nan
    t = norm_txt(x)
    for k, v in MESES.items():
        if k in t:
            return v
    try:
        return pd.to_datetime(x, dayfirst=True).month
    except Exception:
        return np.nan

med["mes_num"] = med[C["mes"]].map({v: mes_a_num(v) for v in med[C["mes"]].unique()})
med["fecha_cierre_dt"] = pd.to_datetime(med[C["cierre"]], errors="coerce", dayfirst=True)
print("Meses:", med[C["mes"]].dropna().unique().tolist())
print(f"Con fecha de cierre: {med['fecha_cierre_dt'].notna().sum()} de {len(med)}")

def estado_simple(x):
    t = norm_txt(x)
    if not isinstance(t, str):
        return "sin dato"
    if t.startswith("no desab"):
        return "no desabastecido"
    if t.startswith("desab"):        # incluye el typo "Desabactecido"
        return "desabastecido"
    if "riesgo" in t:
        return "en riesgo"
    if "monitor" in t:
        return "en monitorización"
    return "otro"

med["estado_simple"] = med[C["estado"]].map(estado_simple)
med["desabastecido"] = med["estado_simple"].eq("desabastecido")

print("\nEstado:")
tabla(med["estado_simple"].value_counts(dropna=False).rename_axis("estado").reset_index(name="registros"))
print("\nPrioridad en Salud:")
tabla(med[C["prioridad"]].value_counts(dropna=False).rename_axis("prioridad").reset_index(name="registros"))
print("\nCausas más frecuentes:")
tabla(med[C["causa"]].value_counts(dropna=False).head(10).rename_axis("causa").reset_index(name="registros"))

Medicamentos:
   duplicados eliminados: 0
Columnas: {'mes': 'mes', 'farmaco': 'nombre_del_farmaco', 'estado': 'estado', 'causa': 'causa', 'cierre': 'fecha_de_cierre', 'indicacion': 'indicaciony_o_tratamiento', 'prioridad': 'prioridad_en_salud'}
Meses: ['Agosto']
Con fecha de cierre: 0 de 237

Estado:


estado,registros
en monitorización,148
no desabastecido,39
desabastecido,32
en riesgo,18



Prioridad en Salud:


prioridad,registros
ENFERMEDADES CRÓNICAS,91
ENFERMEDADES INFECCIOSAS,63
CÁNCER,23
SALUD SEXUAL Y REPRODUCTIVA,18
SALUD MENTAL,17
USO ANESTÉSICO,7
VASOACTIVO / INOTRÓPICOS,6
LÍQUIDOS Y ELECTROLITOS,5
MEDICAMENTO USO DIAGNÓSTICO,5
VARIOS,2



Causas más frecuentes:


causa,registros
Aumento de la demanda,74
No definida por el INVIMA.,47
Problemas en la adquisición de materia prima,25
Problemas de manufactura,21
Insuficientes oferentes,19
Ninguna. Medicamento no desabastecido,17
Baja rentabilidad,9
Alerta sanitaria / Descontinuación del producto,6
Modificaciones de Registro Sanitario pendientes,6
aumento de la demanda,2


In [5]:
# Prioritario = Prioridad en Salud crónicas o cáncer (confirmar con el equipo)
PRIORIDADES_ESTUDIO = {"enfermedades cronicas", "cancer"}

def es_prioritario(x):
    return norm_txt(x) in PRIORIDADES_ESTUDIO

med["prioritario"] = med[C["prioridad"]].map(es_prioritario)
print("Prioritarios:", int(med["prioritario"].sum()), "de", len(med))

Prioritarios: 114 de 237


### 2.2 Clasificación en 4 grupos de enfermedad
Cáncer → neoplasias. Crónicas → palabras clave en indicación y nombre. Varias coincidencias = *ambiguo*; ninguna = *otros*.

In [6]:
GRUPOS = {
    "cardiocerebrovascular": ["hipertens", "cardiac", "cardio", "angina", "infarto", "anticoag", "trombo", "antiagreg",
                              "warfarina", "enoxaparina", "heparina", "estatina", "losartan", "enalapril", "amlodipino",
                              "carvedilol", "metoprolol", "furosemida", "espironolactona", "digoxina", "clopidogrel",
                              "ictus", "arritmia", "antiarritm", "cerebrovascular", "diuretic", "betabloq",
                              "antagonistas del calcio", "canales de calcio", "renina", "angiotens", "hipolipemi",
                              "lipid", "vasodilat", "nitroglicerina", "isosorbide", "adenosina", "amiodarona",
                              "hidralazina", "nifedipin", "canales calcio"],
    "neoplasias": ["cancer", "oncolog", "neoplasi", "tumor", "leucemia", "linfoma", "mieloma", "quimioterap",
                   "antineopl", "carcinoma", "metastas", "tamoxifeno", "capecitabina", "imatinib", "rituximab",
                   "trastuzumab", "cisplatino", "carboplatino", "paclitaxel", "doxorrubicina", "antiandrogen"],
    "respiratorias": ["asma", "antiasm", "epoc", "broncodil", "salbutamol", "ipratropio", "budesonida", "formoterol",
                      "tiotropio", "beclometasona", "fluticasona", "salmeterol", "montelukast", "obstructivo",
                      "bronquitis", "pulmonar", "respirator"],
    "diabetes": ["diabetes", "antidiabet", "insulin", "metformina", "glibenclamida", "glimepirida", "sitagliptina",
                 "linagliptina", "empagliflozina", "dapagliflozina", "hiperglucemia", "glucemia", "hipoglucemiante",
                 "glargina", "liraglutida", "vildagliptina"],
}
PALABRA_COMPLETA = {"asma", "epoc", "ictus", "angina", "tumor"}   # evita "plasma" = "asma"

def _coincide(k, texto):
    if k in PALABRA_COMPLETA:
        return re.search(r"\b" + re.escape(k) + r"\b", texto) is not None
    return k in texto

def grupos_de(texto):
    return [g for g, claves in GRUPOS.items() if any(_coincide(k, texto) for k in claves)]

med["_texto"] = (med[C["indicacion"]].astype(str) + " " + med[C["farmaco"]].astype(str)).map(norm_txt)
med["_prio"] = med[C["prioridad"]].map(norm_txt)
_cache = {t: grupos_de(t) for t in med["_texto"].unique()}
hits_kw = med["_texto"].map(_cache)

# Excepciones manuales por nombre (la indicación de la fuente no coincide con el uso real)
EXCEPCIONES = {"aminofilina": "respiratorias"}

def _hit(prio, h, farmaco):
    if prio == "cancer":
        return ["neoplasias"]
    e = next((g for k, g in EXCEPCIONES.items() if k in norm_txt(farmaco)), None)
    return [e] if e else h

med["grupos_hit"] = [_hit(p, h, f) for p, h, f in zip(med["_prio"], hits_kw, med[C["farmaco"]])]
med["grupo"] = [
    ("no_prioritario" if not pr else (h[0] if len(h) == 1 else ("ambiguo" if len(h) > 1 else "otros")))
    for h, pr in zip(med["grupos_hit"], med["prioritario"])
]
tabla(med["grupo"].value_counts().rename_axis("grupo").reset_index(name="registros"))
print(f"\nPrioritarios en los 4 grupos: {med.loc[med['prioritario'], 'grupo'].isin(GRUPOS).mean() * 100:.1f}%")

grupo,registros
no_prioritario,123
otros,52
neoplasias,24
cardiocerebrovascular,24
diabetes,11
respiratorias,3



Prioritarios en los 4 grupos: 54.4%


In [7]:
# Revisión 1: prioritarios sin grupo (agregar palabras clave y repetir la celda anterior)
revisar = (med.loc[med["prioritario"] & ~med["grupo"].isin(GRUPOS),
                   [C["mes"], C["farmaco"], C["indicacion"], "estado_simple", "grupo"]]
           .sort_values(C["indicacion"]).reset_index(drop=True))
print(f"Prioritarios sin grupo: {len(revisar)}")
tabla(revisar)
os.makedirs("outputs", exist_ok=True)
revisar.to_csv("outputs/medicamentos_por_revisar.csv", index=False, encoding="utf-8-sig")

# Revisión 2: no prioritarios que parecen de los 4 grupos
omisiones = (med.loc[~med["prioritario"] & hits_kw.map(len).gt(0),
                     [C["mes"], C["farmaco"], C["indicacion"], C["prioridad"]]].reset_index(drop=True))
print(f"\nNo prioritarios que parecen de los 4 grupos: {len(omisiones)}")
tabla(omisiones)

Prioritarios sin grupo: 52


mes,fármaco,indicación o tratamiento,estado_simple,grupo
Agosto,Oxicodona tableta y solución inyectable.,ANALGESICOS OPIACEOS,desabastecido,otros
Agosto,Morfina 10 mg/ml Solución Inyectable,ANALGESICOS OPIACEOS,en monitorización,otros
Agosto,"Hidromorfona tabletas 2mg, 2,5mg y 5 mg",ANALGESICOS OPIACEOS,en monitorización,otros
Agosto,Eritropoyetina polvo estéril para reconstituir a suspensión inyectable,ANTIANEMICOS,en riesgo,otros
Agosto,Sacarato de Hierro III 20mg/ ml,ANTIANEMICOS,en monitorización,otros
Agosto,Eritropoyetina recombinante humana solución inyectable,ANTIANEMICOS,no desabastecido,otros
Agosto,Cefuroxima tabletas,ANTIBACTERIANOS DE USO SISTEMICO,en monitorización,otros
Agosto,Biperideno tableta 2mg,ANTICOLINERGICOS (ANTIPARKINSONIANOS),en monitorización,otros
Agosto,Imipramina tableta,ANTIDEPRESIVOS,en monitorización,otros
Agosto,Mirtazapina tableta de 15 mg y 30mg,ANTIDEPRESIVOS,en monitorización,otros



No prioritarios que parecen de los 4 grupos: 8


mes,fármaco,indicación o tratamiento,prioridad en salud
Agosto,Adenosina 3 mg y 6 mg,AGENTES DE ACCION DIRECTA CONTRA EL VIRUS (SISTEM),ENFERMEDADES INFECCIOSAS
Agosto,Adrenalina solución inyectable,"ESTIMULANTES CARDIACOS, EXCLUIDOS GLICOSIDOS CARD.",VASOACTIVO / INOTRÓPICOS
Agosto,Dinitrato de isosorbida 5 mg Tableta sublingual,VASODILATADORES USADOS EN CARDIOTERAPIA,VASOACTIVO / INOTRÓPICOS
Agosto,Dobutamina solución inyectable,"ESTIMULANTES CARDIACOS, EXCLUIDOS GLICOSIDOS CARD.",VASOACTIVO / INOTRÓPICOS
Agosto,Extracto de surfactante pulmonar bovino 3ml,TRATAMIENTO NEONATOS,SALUD SEXUAL Y REPRODUCTIVA
Agosto,"Isoproterenol 0,2 mg/ml Inyectables","ESTIMULANTES CARDIACOS, EXCLUIDOS GLICOSIDOS CARD.",VASOACTIVO / INOTRÓPICOS
Agosto,Norepinefrina solución inyectable 1mg/1mL,"ESTIMULANTES CARDIACOS, EXCLUIDOS GLICOSIDOS CARD.",VASOACTIVO / INOTRÓPICOS
Agosto,Propafenona tableta 150mg,ANTIARRITMICOS,VASOACTIVO / INOTRÓPICOS


### 2.3 Indicadores de medicamentos por grupo
Conteos de medicamentos distintos. H2 solo se calcula si hay fecha de cierre.

In [8]:
med4 = med[med["grupo"].isin(GRUPOS)].copy()

def n_unicos(d, mask):
    return d.loc[mask, C["farmaco"]].nunique()

filas = []
for g in GRUPOS:
    d = med4[med4["grupo"] == g]
    filas.append({
        "grupo": g,
        "n_medicamentos_listados": d[C["farmaco"]].nunique(),
        "n_desabastecidos": n_unicos(d, d["desabastecido"]),
        "n_en_riesgo": n_unicos(d, d["estado_simple"].eq("en riesgo")),
        "n_en_monitorización": n_unicos(d, d["estado_simple"].eq("en monitorización")),
    })
ind_med = pd.DataFrame(filas)
ind_med["pct_desabastecidos"] = np.where(ind_med["n_medicamentos_listados"] > 0,
                                         (ind_med["n_desabastecidos"] / ind_med["n_medicamentos_listados"] * 100).round(1), np.nan)
tabla(ind_med)

# H2: duración (mes de inicio a fecha de cierre)
_ini = pd.to_datetime(med[C["mes"]], errors="coerce", dayfirst=True)
med["duracion_dias"] = (med["fecha_cierre_dt"] - _ini).dt.days
if med["duracion_dias"].notna().any():
    tabla(med.groupby("prioritario")["duracion_dias"].agg(["count", "mean", "median"]).reset_index())
else:
    print("H2 no calculable: sin fecha de cierre ni de inicio completa.")

grupo,n_medicamentos_listados,n_desabastecidos,n_en_riesgo,n_en_monitorización,pct_desabastecidos
cardiocerebrovascular,23,4,1,15,17.4
neoplasias,24,3,2,14,12.5
respiratorias,3,0,0,2,0.0
diabetes,11,2,1,7,18.2


H2 no calculable: sin fecha de cierre ni de inicio completa.


### 2.3.1 KPIs

In [9]:
total = len(med)
n_desab = int(med["desabastecido"].sum())
n_alerta = int(med["estado_simple"].isin(["en riesgo", "en monitorización"]).sum())
n_prior = int(med["prioritario"].sum())
n_prior_desab = int((med["prioritario"] & med["desabastecido"]).sum())
pct_clas = med.loc[med["prioritario"], "grupo"].isin(GRUPOS).mean() * 100
n_cierre = int(med["fecha_cierre_dt"].notna().sum())

kpis = pd.DataFrame([
    ("Registros en el listado", f"{total:,}"),
    ("En desabastecimiento efectivo", f"{n_desab:,} ({n_desab / total * 100:.1f}%)"),
    ("En riesgo o monitorización", f"{n_alerta:,} ({n_alerta / total * 100:.1f}%)"),
    ("Prioritarios (crónicas y cáncer)", f"{n_prior:,} ({n_prior / total * 100:.1f}%)"),
    ("Prioritarios desabastecidos", f"{n_prior_desab:,}"),
    ("Prioritarios en los 4 grupos de estudio", f"{pct_clas:.1f}%"),
    ("Registros con fecha de cierre", f"{n_cierre} de {total}"),
], columns=["Indicador", "Valor"])
tabla(kpis, ancho=420)
kpis.to_csv("outputs/kpis.csv", index=False, encoding="utf-8-sig")

Indicador,Valor
Registros en el listado,237
En desabastecimiento efectivo,32 (13.5%)
En riesgo o monitorización,166 (70.0%)
Prioritarios (crónicas y cáncer),114 (48.1%)
Prioritarios desabastecidos,15
Prioritarios en los 4 grupos de estudio,54.4%
Registros con fecha de cierre,0 de 237


### 2.4 Mortalidad: formato común
Los archivos traen **casos** (no tasas). La edad quinquenal solo sirve para filtrar 60-69 y no se conserva.
Se mantienen año, sexo, nacionalidad, localidad y CIE-10.

In [10]:
def tidy_mortalidad(df, grupo, edad_min=60, edad_max=69):
    print(f"{grupo}:")
    d = limpiar(df)
    col = {
        "ano": "ano" if "ano" in d.columns else buscar_col(d, "anio"),
        "sexo": buscar_col(d, "sexo"),
        "nacionalidad": buscar_col(d, "nacionalidad"),
        "edad": buscar_col(d, "edad"),
        "cie10": buscar_col(d, "cie10"),
        "localidad": buscar_col(d, "localidad"),
        "casos": buscar_col(d, "casos"),
    }
    faltan = [k for k, v in col.items() if v is None]
    if faltan:
        raise ValueError(f"{grupo}: falta {faltan}. Columnas: {list(d.columns)}")
    # Filtro de edad 60-69 (primer número del rango)
    ini = pd.to_numeric(d[col["edad"]].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    d = d[(ini >= edad_min) & (ini <= edad_max)]
    print("   edades usadas:", sorted(d[col["edad"]].dropna().unique()))
    out = pd.DataFrame({
        "grupo": grupo,
        "anio": pd.to_numeric(d[col["ano"]], errors="coerce"),
        "sexo": d[col["sexo"]],
        "nacionalidad": d[col["nacionalidad"]],
        "localidad": d[col["localidad"]],
        "cie10": d[col["cie10"]],
        "casos": pd.to_numeric(d[col["casos"]], errors="coerce"),
    }).dropna(subset=["anio", "casos"])
    out["anio"] = out["anio"].astype(int)
    out = out.groupby(["grupo", "anio", "sexo", "nacionalidad", "localidad", "cie10"],
                      as_index=False, dropna=False)["casos"].sum()
    print(f"   filas={len(out)} | casos={int(out['casos'].sum())}")
    return out

claves_mort = {"cardiocerebrovascular": "cardio", "neoplasias": "neoplasias",
               "respiratorias": "respiratorias", "diabetes": "diabetes"}
partes = [tidy_mortalidad(datos[k], g) for g, k in claves_mort.items() if k in datos]
if not partes:
    raise ValueError("Sin datos de mortalidad en data/raw")
mort = pd.concat(partes, ignore_index=True)

# Total Bogotá por grupo y año; revisar cobertura y años incompletos
mort_bogota = mort.groupby(["grupo", "anio"], as_index=False)["casos"].sum()
tabla(mort_bogota.pivot(index="anio", columns="grupo", values="casos").rename_axis(columns=None).reset_index())

cardiocerebrovascular:
   duplicados eliminados: 0
   edades usadas: ['De 60 a 64 años', 'De 65 a 69 años']
   filas=248 | casos=744
neoplasias:
   duplicados eliminados: 0
   edades usadas: ['De 60 a 64 años', 'De 65 a 69 años']
   filas=2859 | casos=8009
respiratorias:
   duplicados eliminados: 0
   edades usadas: ['De 60 a 64 años', 'De 65 a 69 años']
   filas=517 | casos=1235
diabetes:
   duplicados eliminados: 0
   edades usadas: ['De 60 a 64 años', 'De 65 a 69 años']
   filas=1008 | casos=1668


año,cardiocerebrovascular,diabetes,neoplasias,respiratorias
2019,nan,209.0,994.0,177.0
2020,nan,261.0,960.0,140.0
2021,nan,230.0,977.0,138.0
2022,nan,195.0,995.0,137.0
2023,nan,189.0,1060.0,161.0
2024,nan,205.0,1113.0,173.0
2025,nan,256.0,1213.0,201.0
2026,744.0,123.0,697.0,108.0


## 3. Integración
Llave: `grupo`. Mortalidad en el último año común a los 4 grupos. Son casos, no tasas: sirven para ordenar la carga entre grupos. Con 4 grupos el ranking es descriptivo.

In [11]:
comunes = set.intersection(*[set(v) for v in mort_bogota.groupby("grupo")["anio"].apply(list)])
if not comunes:
    raise ValueError("Los 4 grupos no comparten año")
ANIO_REF = max(comunes)
ANIO_PARCIAL = int(mort_bogota["anio"].max())   # año en curso: datos incompletos
print("Año de referencia:", ANIO_REF, "| año parcial:", ANIO_PARCIAL)

ref = mort_bogota[mort_bogota["anio"] == ANIO_REF][["grupo", "casos"]].rename(columns={"casos": "muertes_60_69"})
cruce = ind_med.merge(ref, on="grupo", how="left")
cruce["anio_ref"] = ANIO_REF
cruce["rank_mortalidad"] = cruce["muertes_60_69"].rank(ascending=False, method="min")
cruce["rank_desabastecimiento"] = cruce["n_desabastecidos"].rank(ascending=False, method="min")
cruce[["rank_mortalidad", "rank_desabastecimiento"]] = cruce[["rank_mortalidad", "rank_desabastecimiento"]].astype("Int64")
tabla(cruce)
print("Spearman entre rankings (descriptivo, n=4):",
      round(cruce["rank_mortalidad"].corr(cruce["rank_desabastecimiento"]), 2))

# Lectura territorial y por sexo (año de referencia)
ref_mort = mort[mort["anio"] == ANIO_REF]
es_sin_loc = ref_mort["localidad"].astype(str).str.startswith("99")   # "99 - Sin Información"
print(f"Casos sin localidad (fuera del análisis territorial): {int(ref_mort.loc[es_sin_loc, 'casos'].sum())}")
terr = ref_mort[~es_sin_loc].groupby(["localidad", "grupo"], as_index=False)["casos"].sum()
sexo = ref_mort.groupby(["grupo", "sexo"], as_index=False)["casos"].sum()
print("\nLocalidades con más muertes prematuras (H3):")
tabla(terr.groupby("localidad")["casos"].sum().sort_values(ascending=False).head(10).reset_index())

Año de referencia: 2026 | año parcial: 2026


grupo,n_medicamentos_listados,n_desabastecidos,n_en_riesgo,n_en_monitorización,pct_desabastecidos,muertes (60-69 años),año_ref,rank_mortalidad,rank_desabastecimiento
cardiocerebrovascular,23,4,1,15,17.4,744,2026,1,1
neoplasias,24,3,2,14,12.5,697,2026,2,2
respiratorias,3,0,0,2,0.0,108,2026,4,4
diabetes,11,2,1,7,18.2,123,2026,3,3


Spearman entre rankings (descriptivo, n=4): 1.0
Casos sin localidad (fuera del análisis territorial): 24

Localidades con más muertes prematuras (H3):


localidad,casos
11 - Suba,229
08 - Kennedy,217
10 - Engativá,174
07 - Bosa,159
19 - Ciudad Bolívar,148
01 - Usaquén,107
04 - San Cristóbal,95
18 - Rafael Uribe Uribe,88
05 - Usme,83
09 - Fontibón,64


## 4. Visualizaciones interactivas
Pasar el mouse para ver el detalle. Zoom, arrastre y clics en leyenda desactivados. No se guardan imágenes.

In [12]:
COLORES = {"cardiocerebrovascular": "#4C78A8", "neoplasias": "#F58518",
           "respiratorias": "#54A24B", "diabetes": "#B279A2"}

FIGS = {}   # figuras guardadas para armar el dashboard HTML

def mostrar(fig, altura=480, nombre=None):
    # Gráfica de solo lectura: detalle al pasar el mouse
    if nombre:
        FIGS[nombre] = fig
    fig.update_layout(dragmode=False, height=altura, margin=dict(l=60, r=30, t=70, b=60),
                      legend=dict(itemclick=False, itemdoubleclick=False), hoverlabel=dict(font_size=14))
    fig.update_xaxes(fixedrange=True)
    fig.update_yaxes(fixedrange=True)
    fig.show(config={"displayModeBar": False, "scrollZoom": False, "doubleClick": False, "showTips": False})

In [13]:
# 4.1 Brecha: mortalidad vs desabastecimiento por grupo
dat = cruce.dropna(subset=["muertes_60_69"])
fig = px.scatter(
    dat, x="muertes_60_69", y="n_desabastecidos", color="grupo", text="grupo", color_discrete_map=COLORES,
    hover_data={"n_medicamentos_listados": True, "n_en_riesgo": True, "n_en_monitorización": True, "grupo": False},
    labels={"muertes_60_69": "Muertes prematuras (60-69 años)", "n_desabastecidos": "Medicamentos prioritarios desabastecidos",
            "n_medicamentos_listados": "Medicamentos listados", "n_en_riesgo": "En riesgo",
            "n_en_monitorización": "En monitorización"},
    title="Mortalidad prematura vs. desabastecimiento por grupo de enfermedad")
fig.update_traces(marker=dict(size=24), textposition="top center")
fig.update_layout(showlegend=False, yaxis=dict(rangemode="tozero", dtick=1), xaxis=dict(rangemode="tozero"))
mostrar(fig, nombre="brecha")

In [14]:
# 4.2 Estado de los medicamentos prioritarios por grupo
ORDEN = ["desabastecido", "en riesgo", "en monitorización", "no desabastecido"]
est = med4.groupby(["grupo", "estado_simple"])[C["farmaco"]].nunique().reset_index(name="medicamentos")
fig = px.bar(est, x="grupo", y="medicamentos", color="estado_simple", barmode="stack",
             category_orders={"estado_simple": ORDEN, "grupo": list(GRUPOS)},
             color_discrete_map={"desabastecido": "#E45756", "en riesgo": "#F58518",
                                 "en monitorización": "#EECA3B", "no desabastecido": "#9D9DA1"},
             labels={"grupo": "", "medicamentos": "Medicamentos distintos", "estado_simple": "Estado"},
             title="Estado de los medicamentos prioritarios por grupo")
mostrar(fig, nombre="estado")

In [15]:
# 4.3 Causas más frecuentes por grupo
cau = (med4.assign(causa=med4[C["causa"]].fillna("sin dato").astype(str).str[:40])
           .groupby(["grupo", "causa"]).size().reset_index(name="registros"))
top = cau.groupby("causa")["registros"].sum().nlargest(6).index
fig = px.bar(cau[cau["causa"].isin(top)], y="grupo", x="registros", color="causa", orientation="h", barmode="stack",
             labels={"grupo": "", "registros": "Registros", "causa": "Causa"},
             title="Causas más frecuentes por grupo")
mostrar(fig, nombre="causas")

In [16]:
# 4.4 Evolución por grupo: línea = años completos; rombo = año en curso (parcial, no comparable)
comp = mort_bogota[mort_bogota["anio"] < ANIO_PARCIAL].sort_values("anio")
parc = mort_bogota[mort_bogota["anio"] == ANIO_PARCIAL]
fig = px.line(comp, x="anio", y="casos", color="grupo", markers=True, color_discrete_map=COLORES,
              labels={"anio": "Año", "casos": "Muertes prematuras (60-69 años)", "grupo": "Grupo"},
              title=f"Evolución de la mortalidad prematura (◇ = {ANIO_PARCIAL}, año parcial)")
extra = px.scatter(parc, x="anio", y="casos", color="grupo", color_discrete_map=COLORES,
                   labels={"anio": "Año", "casos": "Muertes prematuras (60-69 años)", "grupo": "Grupo"})
extra.update_traces(marker=dict(symbol="diamond-open", size=14, line=dict(width=2)), showlegend=False)
fig.add_traces(extra.data)
fig.update_xaxes(dtick=1)
mostrar(fig, nombre="evolucion")

In [17]:
# 4.5 Lectura territorial: muertes prematuras por localidad (H3)
fig = px.bar(terr, y="localidad", x="casos", color="grupo", orientation="h", barmode="stack",
             color_discrete_map=COLORES,
             labels={"localidad": "", "casos": "Muertes prematuras (60-69 años)", "grupo": "Grupo"},
             title="Muertes prematuras por localidad y grupo")
fig.update_yaxes(categoryorder="total ascending")
mostrar(fig, altura=700, nombre="localidad")

In [18]:
# 4.6 Enfoque de género: muertes prematuras por sexo
fig = px.bar(sexo, x="grupo", y="casos", color="sexo", barmode="group",
             labels={"grupo": "", "casos": "Muertes prematuras (60-69 años)", "sexo": "Sexo"},
             title="Muertes prematuras por grupo y sexo")
mostrar(fig, nombre="sexo")

## 5. Exportar

In [19]:
cols_med = [C["mes"], "mes_num", C["farmaco"], C["indicacion"], C["estado"], "estado_simple", C["causa"],
            C["cierre"], C["prioridad"], "prioritario", "desabastecido", "grupo"]
for c in ["data/processed", "outputs"]:
    os.makedirs(c, exist_ok=True)
med[cols_med].to_csv("data/processed/medicamentos_clasificados.csv", index=False, encoding="utf-8-sig")
mort.rename(columns=BONITO).to_csv("data/processed/mortalidad_tidy.csv", index=False, encoding="utf-8-sig")           # detalle 60-69
cruce.rename(columns=BONITO).to_csv("outputs/indicadores_por_grupo.csv", index=False, encoding="utf-8-sig")
mort_bogota.rename(columns=BONITO).to_csv("outputs/mortalidad_bogota.csv", index=False, encoding="utf-8-sig")         # grupo, año
terr.to_csv("outputs/mortalidad_localidad.csv", index=False, encoding="utf-8-sig")             # grupo, localidad
sexo.to_csv("outputs/mortalidad_sexo.csv", index=False, encoding="utf-8-sig")                  # grupo, sexo
with open("requirements.txt", "w") as f:
    f.write("pandas\nnumpy\nrequests\nplotly\nnbformat\nopenpyxl\n")
print("Listo: data/processed y outputs.")

Listo: data/processed y outputs.


### 5.1 Excel ordenado (para leer)
Los CSV son para código. `resultados.xlsx` trae lo mismo ya ordenado: una hoja por tema, encabezados claros, columnas anchas y filtros.

In [20]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

evol = (mort_bogota.pivot(index="anio", columns="grupo", values="casos")
        .rename_axis(columns=None).reset_index().rename(columns={"anio": "año"}))
loc = terr.pivot_table(index="localidad", columns="grupo", values="casos", aggfunc="sum", fill_value=0)
loc["total"] = loc.sum(axis=1)
loc = loc.sort_values("total", ascending=False).reset_index()
sex = sexo.pivot(index="grupo", columns="sexo", values="casos").rename_axis(columns=None).reset_index()
meds = med[[C["mes"], C["farmaco"], C["indicacion"], "estado_simple", C["causa"], C["prioridad"], "prioritario",
            "desabastecido", "grupo"]].copy()
meds["prioritario"] = meds["prioritario"].map({True: "Sí", False: "No"})
meds["desabastecido"] = meds["desabastecido"].map({True: "Sí", False: "No"})

HOJAS = {"Resumen por grupo": cruce, "Evolución": evol, "Por localidad": loc, "Por sexo": sex,
         "Medicamentos": meds, "Indicadores": kpis}

with pd.ExcelWriter("outputs/resultados.xlsx", engine="openpyxl") as w:
    for nombre, d in HOJAS.items():
        d = d.rename(columns=BONITO)
        d.columns = [str(c).replace("_", " ").capitalize() for c in d.columns]
        d.to_excel(w, sheet_name=nombre, index=False)
        ws = w.sheets[nombre]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for j, col in enumerate(d.columns, 1):
            largo = max([len(str(col))] + [len(str(v)) for v in d.iloc[:, j - 1].head(500)])
            ws.column_dimensions[get_column_letter(j)].width = min(largo + 3, 60)
        for celda in ws[1]:
            celda.font = Font(bold=True, color="FFFFFF")
            celda.fill = PatternFill("solid", fgColor="1F4E79")
            celda.alignment = Alignment(vertical="center", wrap_text=True)
        for fila in ws.iter_rows(min_row=2):
            for celda in fila:
                celda.alignment = Alignment(vertical="top", wrap_text=True)
print("Listo: outputs/resultados.xlsx")

Listo: outputs/resultados.xlsx


### 5.2 Dashboard HTML
Página única con KPIs y las gráficas interactivas. Se abre en cualquier navegador y se publica con GitHub Pages.

In [21]:
import html as _html

CONFIG = {"displayModeBar": False, "scrollZoom": False, "doubleClick": False, "responsive": True}
MES = ", ".join(str(m) for m in med[C["mes"]].dropna().unique())

BLOQUES = [
    ("brecha", "La brecha: mortalidad vs. desabastecimiento",
     "Cada punto es un grupo de enfermedad. A la derecha, más muertes prematuras; arriba, más medicamentos prioritarios desabastecidos."),
    ("estado", "Estado de los medicamentos prioritarios",
     "Medicamentos distintos de cada grupo según su estado en el listado."),
    ("causas", "Causas reportadas",
     "Las causas más frecuentes que reporta la Secretaría de Salud por grupo."),
    ("evolucion", "Evolución en el tiempo",
     f"La línea muestra los años completos. El rombo es {ANIO_PARCIAL}, año parcial: no es comparable con los anteriores."),
    ("localidad", "Lectura territorial",
     "Muertes prematuras por localidad. Las de arriba son las de mayor prioridad para el monitoreo."),
    ("sexo", "Enfoque de género",
     "Muertes prematuras por grupo de enfermedad y sexo."),
]

tarjetas = "".join(
    f'<div class="k"><div class="kv">{_html.escape(str(r.Valor))}</div><div class="kt">{_html.escape(str(r.Indicador))}</div></div>'
    for r in kpis.itertuples())

secciones, primera = "", True
for clave, titulo, nota in BLOQUES:
    if clave not in FIGS:
        continue
    cuerpo = FIGS[clave].to_html(full_html=False, include_plotlyjs="cdn" if primera else False, config=CONFIG)
    primera = False
    secciones += f'<section><h2>{_html.escape(titulo)}</h2><p class="n">{_html.escape(nota)}</p>{cuerpo}</section>'

pagina = f"""<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>DataJam 2026 · Medicamentos y mortalidad prematura en Bogotá</title>
<style>
  body {{ margin:0; background:#f6f7f9; color:#212529; font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif; }}
  header {{ background:#1F4E79; color:#fff; padding:28px 20px; }}
  header h1 {{ margin:0 0 6px; font-size:1.5rem; }}
  header p {{ margin:0; opacity:.9; }}
  main {{ max-width:1100px; margin:0 auto; padding:20px; }}
  .kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(190px,1fr)); gap:12px; margin-bottom:24px; }}
  .k {{ background:#fff; border-radius:10px; padding:14px 16px; box-shadow:0 1px 3px rgba(0,0,0,.08); }}
  .kv {{ font-size:1.4rem; font-weight:700; color:#1F4E79; }}
  .kt {{ font-size:.85rem; color:#555; margin-top:4px; }}
  section {{ background:#fff; border-radius:10px; padding:18px 18px 8px; margin-bottom:20px; box-shadow:0 1px 3px rgba(0,0,0,.08); }}
  h2 {{ margin:0 0 4px; font-size:1.15rem; }}
  .n {{ margin:0 0 8px; color:#555; font-size:.92rem; }}
  footer {{ font-size:.82rem; color:#555; padding:0 0 30px; line-height:1.5; }}
</style></head><body>
<header><h1>Medicamentos desabastecidos y mortalidad prematura en Bogotá</h1>
<p>Adultos de 60 a 69 años · Cardiocerebrovascular, neoplasias, respiratorias crónicas y diabetes</p></header>
<main>
<div class="kpis">{tarjetas}</div>
{secciones}
<footer>Fuente: Portal de Datos Abiertos Bogotá (Secretaría Distrital de Salud). Las muertes son casos, no tasas.
{ANIO_PARCIAL} es un año parcial. Medicamentos: listado de {_html.escape(MES)}, sin fecha de cierre. Relación descriptiva, no causal.
Equipo TINTOS · Bogotá DataJam 2026.</footer>
</main></body></html>"""

with open("outputs/dashboard.html", "w", encoding="utf-8") as f:
    f.write(pagina)
print("Listo: outputs/dashboard.html")

Listo: outputs/dashboard.html
